# Client Discovery and Tool Invocation

Notebook 01 showed the messages, and notebook 02 built the server. This notebook takes the host's side: the MCP client. A client never guesses what a server offers. It lists the server's primitives first, reads their names, descriptions, and input schemas, and only then uses them.

You will connect to Northstar the way a host does, through the SDK's `Client` over stdio, and use every primitive the server exposes.

## Learning Objectives

By the end of this notebook, you should be able to:

- Explain what a client does when it connects, and how listing primitives differs from that.
- List, read, get, and call a server's primitives through a real stdio client session.
- Handle a tool error that arrives as a result, and pass it on to the model.

## How A Client Connects

When an SDK client connects, it first sends `server/discover` to learn which protocol versions and capabilities the server supports. If the server only knows the older `initialize` handshake from notebook 01's note, the client falls back to it automatically.

This tells the client which *kinds* of primitive exist, but not what they are. Listing them is a separate step: `tools/list`, `resources/list`, and `prompts/list` return the actual names, descriptions, and schemas.

<div align="center">
  <img src="assets/03-client-discovery-result.png" alt="Client discovery result" width="800" style="max-width: 100%; height: auto;" />
  <p><em>Discovery before invocation. Source: repository-authored diagram of the list requests a client sends to the Northstar server</em></p>
</div>

Discovery is a runtime contract. It lets the client choose among tools, resources, and prompts without hardcoding hidden assumptions about the server.

## Open A Client Session

`StdioServerParameters` tells the client which command starts the server. Here, `sys.executable` is the notebook's own Python (the one `uv sync` set up in `.venv`), and `-m support` makes it start the Northstar server from notebook 02, which speaks MCP over its stdin and stdout. You never start the server yourself: `Client` launches that process, connects to it, and closes both when the `async with` block ends. Each client method sends one of the JSON-RPC requests from notebook 01:

| Client method | JSON-RPC method |
|---|---|
| `list_tools()`, `list_resources()`, `list_prompts()` | `tools/list`, `resources/list`, `prompts/list` |
| `read_resource(uri)` | `resources/read` |
| `get_prompt(name, arguments)` | `prompts/get` |
| `call_tool(name, arguments)` | `tools/call` |

Each `Client(...)` block below starts a fresh server process. Results remain reproducible because Northstar reloads fixed JSON fixtures and does not keep mutable state between calls; stateless MCP does not preserve application memory across process restarts. Requests have a ten-second response timeout so a stalled server does not leave a cell waiting indefinitely.

In [ ]:
import json
import sys

from mcp.client import Client
from mcp.client.stdio import StdioServerParameters, stdio_client
from mcp_types import TextContent, TextResourceContents

parameters = StdioServerParameters(command=sys.executable, args=["-m", "support"])

async with Client(stdio_client(parameters), read_timeout_seconds=10) as client:
    tools = await client.list_tools()
    resources = await client.list_resources()
    prompts = await client.list_prompts()

{
    "tools": [tool.name for tool in tools.tools],
    "resources": [str(resource.uri) for resource in resources.resources],
    "prompts": [prompt.name for prompt in prompts.prompts],
}

The host passes each tool's name, description, and input schema to the model. That is all the model knows about the tool when it decides whether, and how, to call it:

In [ ]:
tool = tools.tools[0]
{"name": tool.name, "description": tool.description, "input_schema": tool.input_schema}

Reading a resource fetches context without triggering an action. The data comes back to the host, not to the model, and the host decides whether to add it to the conversation. Here, the resource lists the systems Northstar may access and the permissions (scopes) each one needs:

In [ ]:
async with Client(stdio_client(parameters), read_timeout_seconds=10) as client:
    resource = await client.read_resource("northstar://systems")

resource_content = resource.contents[0]
assert isinstance(resource_content, TextResourceContents)
json.loads(resource_content.text)

Getting a prompt returns ready-made instruction messages, filled in with the arguments. It does not call any external system; the user or host chooses when to use it:

In [ ]:
async with Client(stdio_client(parameters), read_timeout_seconds=10) as client:
    prompt = await client.get_prompt("incident_update", {"ticket_id": "MCP-1002"})

prompt_content = prompt.messages[0].content
assert isinstance(prompt_content, TextContent)
prompt_content.text

Calling the tool runs `create_escalation` in the server process. The result says whether the call failed and carries the structured plan. As in notebook 01, `requires_human_approval` is only a flag; the host has to act on it:

In [ ]:
async with Client(stdio_client(parameters), read_timeout_seconds=10) as client:
    result = await client.call_tool("create_escalation", {"ticket_id": "MCP-1002"})

{"is_error": result.is_error, "structured_content": result.structured_content}

## Tool Errors Arrive As Results

In notebook 02, calling the tool in process with an unknown ticket raised `ToolError`. Through a client, the same failure looks different:

In [ ]:
async with Client(stdio_client(parameters), read_timeout_seconds=10) as client:
    failed = await client.call_tool("create_escalation", {"ticket_id": "MCP-9999"})

error_content = failed.content[0]
assert isinstance(error_content, TextContent)
{"is_error": failed.is_error, "message": error_content.text}

There is no exception raised. The request itself succeeded, and the failure is data inside the result, the tool error from notebook 01. A host must check `is_error` before using a result, and should pass the message on to the model so it can recover, for example by asking the user for the right ticket ID.

## Exercise: Turn Tool Results Into Model Context

A host has to hand every tool result to the model, including failures. Write an async function `result_for_model(client, name, arguments)` that calls a tool and returns a string:

- on success, the structured content as JSON text,
- on a tool error, `"Tool error: "` followed by the error message.

Open one session with `parameters` in an `async with Client(...)` block, and call your function for the known ticket `MCP-1002`, the unknown ticket `MCP-9999`, and missing arguments `{}`. It must not raise for the two failures, and must not return a plan for them.

In [ ]:
# @TODO: Write result_for_model, then call it for MCP-1002, MCP-9999, and missing arguments.

<details>
<summary>Show Reference Solution</summary>

```python
async def result_for_model(client, name, arguments):
    result = await client.call_tool(name, arguments)
    if result.is_error:
        error_content = result.content[0]
        assert isinstance(error_content, TextContent)
        return f"Tool error: {error_content.text}"
    return json.dumps(result.structured_content)


async with Client(stdio_client(parameters), read_timeout_seconds=10) as client:
    outputs = [
        await result_for_model(client, "create_escalation", arguments)
        for arguments in [{"ticket_id": "MCP-1002"}, {"ticket_id": "MCP-9999"}, {}]
    ]

assert not outputs[0].startswith("Tool error")
assert all(output.startswith("Tool error: ") for output in outputs[1:])
outputs
```

</details>

The missing-argument failure comes from the SDK's schema check and the unknown ticket from the tool's own code, as notebook 02 showed. The client sees both the same way: a result with `is_error` set. Passing the message on lets the model fix the arguments or ask the user, instead of the host crashing or the model mistaking an error for a plan.

## Check Your Understanding

1. What does an SDK client do when it connects to a server, and what does listing primitives add on top of that?
2. Calling a tool in process raised `ToolError`, but through a client the same failure came back as a result. Why does MCP report tool failures as results?
3. Does an in-process SDK call test transport behavior?

<details>
<summary>Show Reference Answers</summary>

1. It sends `server/discover` to learn which protocol versions and capabilities the server supports, and falls back to the older `initialize` handshake if the server does not know it. That only says which kinds of primitive exist. `tools/list`, `resources/list`, and `prompts/list` return the actual names, descriptions, and schemas the host needs before it can use anything.
2. So the failure can reach the model. A result with `isError: true` carries a message the model can read and act on, for example by fixing an argument or asking the user. An exception would stop at the client.
3. No. It skips the subprocess, the stdio transport, and the JSON-RPC encoding. Only a client session exercises the real message exchange and process lifecycle.

</details>

## Further Reading

- [Build an MCP Client](https://modelcontextprotocol.io/docs/develop/build-client): The official tutorial that connects a Python client to a server and an LLM.
- [Discovery](https://modelcontextprotocol.io/specification/2026-07-28/server/discover): What `server/discover` returns and when a client should call it.